In [ ]:
!pip install transformers torch
!pip install prettytable
!pip install bioseq
!pip install peft


  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, BartTokenizer, BartForConditionalGeneration
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.nn import CrossEntropyLoss
import pandas as pd
import csv
import bioseq
from torch.nn import Linear



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/MyDrive/eval.csv'

# Open the CSV file in read mode
with open(file_path, 'r', newline='') as csvfile:
    # Create a CSV reader object
    csv_reader = csv.reader(csvfile)

    # Read the headers from the CSV file
    headers = next(csv_reader)

    # Print the headers
    print("Headers in eval.csv:")
    print(headers)


df = pd.read_csv(file_path)

Headers in eval.csv:
['', 'accession', 'name', 'Full Name', 'taxon', 'sequence', 'function', 'AlphaFoldDB']


In [ ]:
# 1. Load Data
# Assuming df is a DataFrame with 'sequence' as input and 'summary' as target
sequences = df['sequence'].tolist()
summaries = df['function'].tolist()

In [ ]:
train_sequences, val_sequences, train_summaries, val_summaries = train_test_split(sequences, summaries, test_size=0.2)


In [ ]:
class SequenceSummaryDataset(Dataset):
    def __init__(self, sequences, summaries, tokenizer, max_length=124):
        self.sequences = sequences
        self.summaries = summaries
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        summary = self.summaries[idx]
        inputs = self.tokenizer(sequence, return_tensors="pt", max_length=self.max_length, truncation=True, padding="max_length")
        targets = self.tokenizer(summary, return_tensors="pt", max_length=self.max_length, truncation=True, padding="max_length")
        return inputs.input_ids.squeeze(), inputs.attention_mask.squeeze(), targets.input_ids.squeeze()


In [ ]:
# Load ESM-2 tokenizer and model
esm2_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm2_model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")

# Load BART tokenizer and model
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
class ESM2BARTWrapper(torch.nn.Module):
    def __init__(self, esm2_model, bart_model, output_dim=768):
        super().__init__()
        self.esm2_model = esm2_model
        self.bart_model = bart_model
        self.projection = Linear(esm2_model.config.hidden_size, output_dim)  # Project ESM-2 outputs to BART's input size

    def forward(self, input_ids, attention_mask, labels=None):
        # Get ESM-2 embeddings
        esm2_outputs = self.esm2_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        embeddings = esm2_outputs.last_hidden_state

        # Project embeddings to BART's expected input size
        projected_embeddings = self.projection(embeddings)

        # Pass projected embeddings to BART
        decoder_inputs = labels if labels is not None else torch.zeros_like(input_ids)
        outputs = self.bart_model(input_ids=decoder_inputs, encoder_outputs=(projected_embeddings,), labels=labels, return_dict=True)
        return outputs


In [ ]:
train_dataset = SequenceSummaryDataset(train_sequences, train_summaries, esm2_tokenizer)
val_dataset = SequenceSummaryDataset(val_sequences, val_summaries, esm2_tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)


In [ ]:
def train_model(model, train_loader, val_loader, epochs=2, lr=1e-4):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        for input_ids, attention_mask, labels in tqdm(train_loader):
            input_ids = input_ids.to(next(model.parameters()).device)
            attention_mask = attention_mask.to(next(model.parameters()).device)
            labels = labels.to(next(model.parameters()).device)
            optimizer.zero_grad()

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = evaluate_model(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss}, Val Loss: {avg_val_loss}")

def evaluate_model(model, val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in val_loader:
            input_ids = input_ids.to(next(model.parameters()).device)
            attention_mask = attention_mask.to(next(model.parameters()).device)
            labels = labels.to(next(model.parameters()).device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

    return total_loss / len(val_loader)


In [ ]:
esm2_bart_model = ESM2BARTWrapper(esm2_model, bart_model)
train_model(esm2_bart_model, train_loader, val_loader)


100%|██████████| 835/835 [2:06:46<00:00,  9.11s/it]


Epoch 1/2, Train Loss: 0.7787779455413362, Val Loss: 0.41437600369444877


100%|██████████| 835/835 [2:05:28<00:00,  9.02s/it]


Epoch 2/2, Train Loss: 0.4014926742204649, Val Loss: 0.3885206641299588


In [ ]:
# Evaluate BLEU Scores
from nltk.translate.bleu_score import sentence_bleu

def evaluate_bleu(model, data_loader):
    model.eval()
    bleu_scores = []
    count = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels in data_loader:
            input_ids = input_ids.to(next(model.parameters()).device)
            attention_mask = attention_mask.to(next(model.parameters()).device)
            labels = labels.to(next(model.parameters()).device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            generated_ids = outputs.logits.argmax(dim=-1)

            for i in range(len(input_ids)):
                reference = bart_tokenizer.decode(labels[i], skip_special_tokens=True)
                candidate = bart_tokenizer.decode(generated_ids[i], skip_special_tokens=True)
                reference_tokens = reference.split()
                candidate_tokens = candidate.split()
                bleu_score = sentence_bleu([reference_tokens], candidate_tokens)
                bleu_scores.append(bleu_score)

                if count < 5:
                    print(f"Actual Summary: {reference}")
                    print(f"Predicted Summary: {candidate}")
                    print(f"BLEU Score: {bleu_score}\n")
                    count += 1

    avg_bleu_score = sum(bleu_scores) / len(bleu_scores)
    print(f"Average BLEU Score: {avg_bleu_score}")

evaluate_bleu(esm2_bart_model, val_loader)

/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Actual Summary:  The by with in with saids
Predicted Summary:  that in
BLEU Score: 2.0732986305800918e-232

Actual Summary:  a by. was ats of. was at---s- the. asss
Predicted Summary: - by at was at-- was
BLEU Score: 8.396161215621529e-232

Actual Summary: , in that by a� the a� the a� thes a� the by by by a� the bys the for� the for� the for� the by by by for� thes
Predicted Summary:  at that� the� the� the� the a a by� the a� the� the� the a a� the
BLEU Score: 3.1821288148979075e-155

Actual Summary: -s-s as a a� thes
Predicted Summary:  a� the
BLEU Score: 3.418291552750845e-232

Actual Summary: - a� thes that a� the a� thes a and the The by a� the and by a� the the a's the a's the a's the a's the a'ss
Predicted Summary:  a� the� the� the� a that a� the a� thes� a's a's a's a's
BLEU Score: 0.07657040892840415

Average BLEU Score: 0.013825221879983175
